In [ ]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

In [ ]:
!pip install torch_geometric

In [ ]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [ ]:
from utils.datasets     import NF_IDS_Dataset
from utils.models       import EdgeGRU_Baseline
from utils.metrics      import calculate_metrics_gnn
from utils.training     import train_epoch, evaluate, find_optimal_threshold
from utils.experiment   import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots


# Functions


## Run multiple seeds

In [14]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

In [15]:
def run_multiple_seeds(model_class, model_config, train_loader, val_loader,
                       manager,
                       seeds=[42, 123, 777, 2024, 99],
                       epochs=60,
                       device='cpu',
                       experiment_name="Exp_Optimized",
                       json_dir="./logs",
                       plots_dir="./plots"):

    os.makedirs(json_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    print(f" Starting Multi-Seed Run: {experiment_name}")
    print(f"   Seeds: {seeds}")
    print("-" * 60)

    for seed in seeds:
        t0_seed = time.perf_counter()
        t_train_total = 0.0
        t_eval_total = 0.0
        t_threshold_total = 0.0

        tz = timezone(timedelta(hours=-3))  # Argentina
        run_ts = datetime.now(tz).strftime("%Y%m%d_%H%M%S")
        run_id = f"{experiment_name}_seed{seed}_{run_ts}"

        print(f"\nRunning seed {seed} | run_id={run_id}")

        # Name
        exp_id = f"{experiment_name}_seed{seed}"
        print(f"\n{exp_id}")

        # Preventive Memory Cleaning (Before loading anything)
        gc.collect()
        torch.cuda.empty_cache()

        set_seed(seed)

        # Update model_config
        model_config['model_name'] = exp_id
        model_config.setdefault("extra_params", {})
        model_config["extra_params"]["run_ts"] = run_ts
        model_config["extra_params"]["run_id"] = run_id

        # Instantiate Model
        model = model_class(**model_config['model_params']).to(device)

        # Training Setup
        optimizer = torch.optim.Adam(model.parameters(), lr=model_config['extra_params']['learning_rate'])
        pos_weight = torch.tensor([model_config['extra_params']['pos_weight']]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        # Early Stopping
        early_stopping = EarlyStopping(patience=10, mode='max', min_delta=0.0001)

        train_hist, val_loss_hist, val_auc_hist = [], [], []

        # Epochs
        for epoch in range(epochs):
            # Detect whether it is a temporal model (ST or GRU) to adjust inputs
            is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

            # --- TRAIN ---
            t0 = time.perf_counter()
            loss_train = train_epoch(model, train_loader, optimizer, criterion,
                                     device=device, is_temporal=is_temporal,
                                     batch_steps=model_config['extra_params']["batch_steps"])
            t_train_total += time.perf_counter() - t0

            # --- EVAL ---
            t0 = time.perf_counter()
            val_loss, y_true, y_probs = evaluate(model, val_loader, criterion, device, is_temporal=is_temporal)
            t_eval_total += time.perf_counter() - t0

            # Monitor: calculate only AUC-PR
            val_auc = average_precision_score(y_true, y_probs)

            # Save history
            train_hist.append(float(loss_train))
            val_loss_hist.append(float(val_loss))
            val_auc_hist.append(float(val_auc))

            # Early Stopping check
            improved = early_stopping(val_auc, model, epoch)

            if improved or (epoch+1) % 10 == 0:
                mark = "(*)" if improved else ""
                print(f"   Ep {epoch+1} | Loss: {loss_train:.4f} | Val Loss: {val_loss:.4f} | Val AUC-PR: {val_auc:.4f} {mark}")

            if early_stopping.early_stop:
                print(f"   Early Stopping in epoch {epoch}")
                break


        # Restoring the best model
        model.load_state_dict(early_stopping.best_model_state)



        # 1. Re-evaluate the BEST model to obtain the clean probs
        _, y_true_best, y_probs_best = evaluate(model, val_loader, criterion, device, is_temporal=is_temporal)

        # 2. We are looking for the optimal threshold (Max Recall subject to Precision)
        t0 = time.perf_counter()
        best_th, _, _ = find_optimal_threshold(
            model, val_loader, device, is_temporal, min_precision=0.90
        )
        t_threshold_total += time.perf_counter() - t0

        # 3. We calculate ALL complex metrics
        final_metrics = calculate_metrics_gnn(y_true_best, y_probs_best, prob_threshold=best_th)
        final_metrics['optimal_threshold'] = best_th
        final_metrics['stopped_epoch'] = len(train_hist)
        final_metrics['seed'] = seed
        final_metrics["run_ts"] = run_ts
        final_metrics["run_id"] = run_id
        final_metrics["time_total_sec"] = time.perf_counter() - t0_seed
        final_metrics["time_train_sec"] = t_train_total
        final_metrics["time_eval_sec"] = t_eval_total
        final_metrics["time_threshold_sec"] = t_threshold_total



        print(f"   Final: Precision={final_metrics['Precision']:.4f} |  Recall={final_metrics['Recall']:.4f} | F1={final_metrics['F1']:.4f} | F2={final_metrics['F2']:.4f} | AUC-PR={final_metrics['AUC-PR']:.4f} | FPR={final_metrics['FPR']:.5f}")

        # 4. Plots and Manager
        save_plots(train_hist, val_loss_hist, y_true_best, y_probs_best, seed, experiment_name, run_ts, save_dir=os.path.join(plots_dir,experiment_name))

        manager.log_experiment(
            model_config=model_config,
            metrics=final_metrics,
            model_object=model
        )

        # Save JSON
        filename_json = os.path.join(
            json_dir, experiment_name,
            f"training_history_{experiment_name}_seed{seed}_{run_ts}.json"
        )

        history_payload = {
            "experiment_name": experiment_name,
            "seed": seed,
            "run_ts": run_ts,
            "run_id": run_id,
            "training": {
                "train_loss": train_hist,
                "val_loss": val_loss_hist,
                "val_aucpr": val_auc_hist
            },
            "early_stopping": {
                "best_epoch": int(early_stopping.best_epoch),
                "best_val_aucpr": float(early_stopping.best_score),
                "stopped_epoch": int(len(train_hist))
            }
        }


        try:
            with open(filename_json, 'w') as f:
                json.dump(history_payload, f, cls=NumpyEncoder, indent=4)
            print(f"Training history saved in: {filename_json}")
        except Exception as e:
            print(f"\nWarning: JSON could not be saved: {e}")


        print(f"\n End seed {seed}")

        print("-" * 60)

    # 8. Print final statistics
    df = pd.read_csv(manager.log_file)
    df_exp = df[df["model_name"].str.contains(experiment_name)]

    if len(df_exp) == 0:
        print("No records found for this experiment.")
    else:

        def mean_std(series):
            return series.mean(), series.std()

        auc_mean, auc_std = mean_std(df_exp["AUC-PR"])
        rec_mean, rec_std = mean_std(df_exp["Recall"])
        ttot_mean, ttot_std = mean_std(df_exp["time_total_sec"])
        ttrain_mean, ttrain_std = mean_std(df_exp["time_train_sec"])

        print("="*50)
        print(f" AVERAGE RESULT: {experiment_name}")
        print(f"AUC-PR:      {auc_mean:.4f} ± {auc_std:.4f}")
        print(f"Recall:      {rec_mean:.4f} ± {rec_std:.4f}")
        print(f"Total Time:  {ttot_mean:.2f} ± {ttot_std:.2f} sec")
        print(f"Train Time:  {ttrain_mean:.2f} ± {ttrain_std:.2f} sec")
        print("="*50)


# Auxiliary

In [16]:
ROOT_PATH = "./dataset_processed"

In [17]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [18]:
ROOT_DIR = "./results_earlystopping"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")

# Main

## Seeds

In [19]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [20]:
model_config = {
    "model_name": None,
    "type": "EdgeGRU_Baseline",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [21]:
EXPERIMENT_NAME="EdgeGRU_BiasOn"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [23]:
run_multiple_seeds(
    model_class=EdgeGRU_Baseline,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: EdgeGRU_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=EdgeGRU_BiasOn_seed42_20260220_115040

EdgeGRU_BiasOn_seed42
   Ep 1 | Loss: 0.2160 | Val Loss: 0.2349 | Val AUC-PR: 0.0724 (*)
   Ep 4 | Loss: 0.2066 | Val Loss: 0.2281 | Val AUC-PR: 0.0740 (*)
   Ep 6 | Loss: 0.2099 | Val Loss: 0.2279 | Val AUC-PR: 0.0742 (*)
   Ep 7 | Loss: 0.2015 | Val Loss: 0.2249 | Val AUC-PR: 0.0752 (*)
   Ep 8 | Loss: 0.2027 | Val Loss: 0.2254 | Val AUC-PR: 0.0803 (*)
   Ep 9 | Loss: 0.2008 | Val Loss: 0.2227 | Val AUC-PR: 0.0844 (*)
   Ep 10 | Loss: 0.2023 | Val Loss: 0.2233 | Val AUC-PR: 0.0756 
   Ep 13 | Loss: 0.1959 | Val Loss: 0.2191 | Val AUC-PR: 0.0946 (*)
   Ep 14 | Loss: 0.1973 | Val Loss: 0.2193 | Val AUC-PR: 0.0957 (*)
   Ep 15 | Loss: 0.1946 | Val Loss: 0.2182 | Val AUC-PR: 0.1057 (*)
   Ep 16 | Loss: 0.1928 | Val Loss: 0.2156 | Val AUC-PR: 0.1180 (*)
   Ep 18 | Loss: 0.1906 | Val Loss: 0